In [1]:
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D, Dense

In [2]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [3]:
base_path = r"C:\Users\Fahad\Desktop\my works\socially relevant Project\tinyML-HAR\data\UCI HAR Dataset"
train_path = os.path.join(base_path, "train", "Inertial Signals")


In [4]:
train_path = os.path.join(base_path, "train", "Inertial Signals")
test_path  = os.path.join(base_path, "test", "Inertial Signals")

In [5]:
acc_x = np.loadtxt(os.path.join(train_path, "total_acc_x_train.txt"))
acc_y = np.loadtxt(os.path.join(train_path, "total_acc_y_train.txt"))
acc_z = np.loadtxt(os.path.join(train_path, "total_acc_z_train.txt"))

In [6]:
gyro_x = np.loadtxt(os.path.join(train_path, "body_gyro_x_train.txt"))
gyro_y = np.loadtxt(os.path.join(train_path, "body_gyro_y_train.txt"))
gyro_z = np.loadtxt(os.path.join(train_path, "body_gyro_z_train.txt"))

In [7]:
X_train = np.stack([acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z], axis=-1)

In [8]:
y_train = np.loadtxt(os.path.join(base_path, "train", "y_train.txt")).astype(int)

In [9]:
print("X_train shape:", X_train.shape)
print("y_train shape:",y_train.shape)

X_train shape: (7352, 128, 6)
y_train shape: (7352,)


In [10]:
print(X_train[0].shape)

(128, 6)


In [11]:
window = X_train[0] 
df = pd.DataFrame(
    window,
    columns=["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
)

df["label"] = y_train[0] 

df.to_csv("window_0.csv", index=False)

In [12]:
df.head()

,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,label
0,1.012817,-0.123217,0.102934,0.030191,0.066014,0.022859,5
1,1.022833,-0.126876,0.105687,0.043711,0.042699,0.010316,5
2,1.022028,-0.124004,0.102102,0.035688,0.074850,0.013250,5
3,1.017877,-0.124928,0.106553,0.040402,0.057320,0.017751,5
4,1.023680,-0.125767,0.102814,0.047097,0.052343,0.002553,5


In [13]:
acc_x_test = np.loadtxt(os.path.join(test_path, "total_acc_x_test.txt"))
acc_y_test = np.loadtxt(os.path.join(test_path, "total_acc_y_test.txt"))
acc_z_test = np.loadtxt(os.path.join(test_path, "total_acc_z_test.txt"))

In [14]:
gyro_x_test = np.loadtxt(os.path.join(test_path, "body_gyro_x_test.txt"))
gyro_y_test = np.loadtxt(os.path.join(test_path, "body_gyro_y_test.txt"))
gyro_z_test = np.loadtxt(os.path.join(test_path, "body_gyro_z_test.txt"))

In [15]:
X_test = np.stack([acc_x_test, acc_y_test, acc_z_test, gyro_x_test, gyro_y_test, gyro_z_test], axis=-1)

In [16]:
y_test = np.loadtxt(os.path.join(base_path, "test", "y_test.txt")).astype(int)

In [17]:
print("X_test shape:", X_test.shape)
print("y_test shape:",y_test.shape)

X_test shape: (2947, 128, 6)
y_test shape: (2947,)


In [18]:
X = np.concatenate([X_train, X_test], axis=0)
y = np.concatenate([y_train, y_test], axis=0)

In [19]:
print("X shape:", X.shape)
print("y shape:",y.shape)

X shape: (10299, 128, 6)
y shape: (10299,)


In [20]:
def map_label(l):
    if l in [1,2,3]: 
        return 0   #walk
    if l == 4: 
        return 1    #sit     
    if l == 5: 
        return 2    #stand 
    if l == 6: 
        return 3     #lying   

y_mapped = np.array([map_label(i) for i in y])

In [21]:
print(y_mapped.shape)

(10299,)


In [22]:
unique, counts = np.unique(y_mapped, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4672), np.int64(1): np.int64(1777), np.int64(2): np.int64(1906), np.int64(3): np.int64(1944)}


In [23]:
y_final = y_mapped.copy()
transition_indices = []

In [24]:
unique, counts = np.unique(y_final, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4672), np.int64(1): np.int64(1777), np.int64(2): np.int64(1906), np.int64(3): np.int64(1944)}


In [25]:
for i in range(1, len(y_mapped)-1):
    if y_mapped[i] != y_mapped[i-1] or y_mapped[i] != y_mapped[i+1]:
        transition_indices.append(i)
        y_final[i] = 4 

In [26]:
unique, counts = np.unique(y_final, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4553), np.int64(1): np.int64(1655), np.int64(2): np.int64(1787), np.int64(3): np.int64(1822), np.int64(4): np.int64(482)}


In [27]:
print("Number of transition windows identified:", len(transition_indices))

Number of transition windows identified: 482


In [28]:
print("First 10 transition indices:", transition_indices[:10])

for idx in transition_indices[:5]:
    print(f"Window {idx}: before={y_mapped[idx-1]}, current={y_mapped[idx]}, next={y_mapped[idx+1]}")

First 10 transition indices: [26, 27, 50, 51, 77, 78, 175, 176, 201, 202]
Window 26: before=2, current=2, next=1
Window 27: before=2, current=1, next=1
Window 50: before=1, current=1, next=3
Window 51: before=1, current=3, next=3
Window 77: before=3, current=3, next=0


In [29]:
print("X shape:", X.shape)
print("y_final shape:", y_final.shape)

X shape: (10299, 128, 6)
y_final shape: (10299,)


In [30]:
unique, counts = np.unique(y_final, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4553), np.int64(1): np.int64(1655), np.int64(2): np.int64(1787), np.int64(3): np.int64(1822), np.int64(4): np.int64(482)}


In [31]:
def generate_not_on_body(num_windows=1000):
    windows = []
    for _ in range(num_windows):
        g = np.random.uniform(-1, 1, size=3)
        g = g / np.linalg.norm(g) * 9.8   
        acc = np.tile(g, (128, 1)) + np.random.normal(0, 0.03, (128, 3))

        gyro = np.random.normal(0, 0.01, (128, 3))

        window = np.concatenate([acc, gyro], axis=1)
        windows.append(window)

    return np.array(windows)

In [32]:
unique, counts = np.unique(y_final, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4553), np.int64(1): np.int64(1655), np.int64(2): np.int64(1787), np.int64(3): np.int64(1822), np.int64(4): np.int64(482)}


In [33]:
not_on_body = generate_not_on_body(1000)

In [34]:
not_on_body.shape

(1000, 128, 6)

In [35]:
labels_nob = np.full((not_on_body.shape[0],), 5)

In [36]:
X = np.concatenate([X, not_on_body], axis=0)
y_final = np.concatenate([y_final, labels_nob], axis=0)

In [37]:
X.shape

(11299, 128, 6)

In [38]:
y_final.shape

(11299,)

In [39]:
print(not_on_body[0][:10])   
print("Label for synthetic:", labels_nob[0])

[[-1.20006504e+00 -8.50010634e+00  4.80298760e+00  6.92683900e-04
  -2.48693601e-03 -6.62121947e-04]
 [-1.19186312e+00 -8.51116334e+00  4.88898431e+00  5.23444148e-03
  -1.05959899e-02 -8.11510562e-03]
 [-1.13415498e+00 -8.44103052e+00  4.78425854e+00 -1.46952323e-02
   3.08970316e-03 -6.72322035e-03]
 [-1.18225441e+00 -8.41840523e+00  4.80541531e+00  4.94242501e-03
  -8.52755863e-03 -6.19717307e-03]
 [-1.16171552e+00 -8.44878803e+00  4.78858742e+00 -9.26195866e-03
  -1.46929549e-02  3.02459718e-03]
 [-1.21506214e+00 -8.45326433e+00  4.84061996e+00  2.53766165e-02
   9.91686064e-03  9.64798783e-03]
 [-1.16635960e+00 -8.46946287e+00  4.84110626e+00  2.36824055e-03
  -7.49653096e-03 -2.42533687e-03]
 [-1.21547448e+00 -8.48337501e+00  4.82162385e+00  5.59949225e-04
   1.98194904e-03 -4.65316265e-03]
 [-1.17415710e+00 -8.40912597e+00  4.86831746e+00  1.71595158e-02
   1.35018179e-02 -5.78878275e-03]
 [-1.21212675e+00 -8.43975999e+00  4.83166285e+00 -1.12306344e-02
  -8.32293445e-03 -1.5169

In [40]:
indices = np.where(y_final == 5)[0]
print("Found", len(indices), "not-on-body windows")

Found 1000 not-on-body windows


In [41]:
unique, counts = np.unique(y_final, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(4553), np.int64(1): np.int64(1655), np.int64(2): np.int64(1787), np.int64(3): np.int64(1822), np.int64(4): np.int64(482), np.int64(5): np.int64(1000)}


In [42]:
indices_5 = np.where(y_final == 5)[0]
print("Number of not-on-body windows:", len(indices_5))

# Print the first one
i = indices_5[0]
print(X[i][:10])   # first 10 samples
print("Label:", y_final[i])
print(X.shape)

Number of not-on-body windows: 1000
[[-1.20006504e+00 -8.50010634e+00  4.80298760e+00  6.92683900e-04
  -2.48693601e-03 -6.62121947e-04]
 [-1.19186312e+00 -8.51116334e+00  4.88898431e+00  5.23444148e-03
  -1.05959899e-02 -8.11510562e-03]
 [-1.13415498e+00 -8.44103052e+00  4.78425854e+00 -1.46952323e-02
   3.08970316e-03 -6.72322035e-03]
 [-1.18225441e+00 -8.41840523e+00  4.80541531e+00  4.94242501e-03
  -8.52755863e-03 -6.19717307e-03]
 [-1.16171552e+00 -8.44878803e+00  4.78858742e+00 -9.26195866e-03
  -1.46929549e-02  3.02459718e-03]
 [-1.21506214e+00 -8.45326433e+00  4.84061996e+00  2.53766165e-02
   9.91686064e-03  9.64798783e-03]
 [-1.16635960e+00 -8.46946287e+00  4.84110626e+00  2.36824055e-03
  -7.49653096e-03 -2.42533687e-03]
 [-1.21547448e+00 -8.48337501e+00  4.82162385e+00  5.59949225e-04
   1.98194904e-03 -4.65316265e-03]
 [-1.17415710e+00 -8.40912597e+00  4.86831746e+00  1.71595158e-02
   1.35018179e-02 -5.78878275e-03]
 [-1.21212675e+00 -8.43975999e+00  4.83166285e+00 -1.12

In [43]:
def add_magnitude_channels(X):
    ax, ay, az = X[:,:,0], X[:,:,1], X[:,:,2]
    gx, gy, gz = X[:,:,3], X[:,:,4], X[:,:,5]

    acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
    gyro_mag = np.sqrt(gx**2 + gy**2 + gz**2)

    X_with_mag = np.concatenate(
        [X, acc_mag[..., np.newaxis], gyro_mag[..., np.newaxis]], axis=2
    )

    return X_with_mag

In [44]:
X_mag = add_magnitude_channels(X)
print(X_mag.shape) 
print(y_final.shape)

(11299, 128, 8)
(11299,)


In [45]:
X_train_raw8, X_test_raw8, y_train, y_test = train_test_split(
    X_mag, y_final, test_size=0.2, random_state=42, shuffle=True
)

In [46]:
X_train_raw6 = X_train_raw8[:, :, :6] 

In [47]:
def random_rotation_matrix():
    # Generate random angles
    theta_x = np.random.uniform(0, 2*np.pi)
    theta_y = np.random.uniform(0, 2*np.pi)
    theta_z = np.random.uniform(0, 2*np.pi)

    # Rotation matrix for X-axis
    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(theta_x), -np.sin(theta_x)],
        [0, np.sin(theta_x),  np.cos(theta_x)]
    ])

    # Rotation matrix for Y-axis
    Ry = np.array([
        [np.cos(theta_y), 0, np.sin(theta_y)],
        [0, 1, 0],
        [-np.sin(theta_y), 0, np.cos(theta_y)]
    ])

    # Rotation matrix for Z-axis
    Rz = np.array([
        [np.cos(theta_z), -np.sin(theta_z), 0],
        [np.sin(theta_z),  np.cos(theta_z), 0],
        [0, 0, 1]
    ])

    # Combined rotation matrix
    R = Rz @ Ry @ Rx
    return R


In [48]:
def rotate_window6_and_add_mags(window6, R):
    # window6 shape = (128, 6)
    acc = window6[:, :3]
    gyro = window6[:, 3:6]
    acc_rot = acc @ R.T
    gyro_rot = gyro @ R.T
    # recompute mags (column vectors)
    acc_mag = np.sqrt((acc_rot**2).sum(axis=1))[:, None]
    gyro_mag = np.sqrt((gyro_rot**2).sum(axis=1))[:, None]
    return np.concatenate([acc_rot, gyro_rot, acc_mag, gyro_mag], axis=1)  # (128,8)


In [49]:
def augment_train_rotations(X6, y, num_aug=1, seed=None):
    if seed is not None:
        np.random.seed(seed)
    X_aug = []
    y_aug = []
    for i in range(len(X6)):
        window6 = X6[i]  # (128,6)
        for _ in range(num_aug):
            R = random_rotation_matrix()
            rotated8 = rotate_window6_and_add_mags(window6, R)
            X_aug.append(rotated8)
            y_aug.append(y[i])
    return np.array(X_aug), np.array(y_aug)  # shapes (num_aug*N_train,128,8), (num_aug*N_train,)

In [50]:
num_aug = 1  # start with 1 rotated copy
X_rot, y_rot = augment_train_rotations(X_train_raw6, y_train, num_aug=num_aug, seed=42)

In [51]:
# 4) prepare original train windows with mags (they already have mags)
X_train_with_mag = X_train_raw8  # (N_train,128,8)

In [52]:
# 5) combine
X_train_combined = np.concatenate([X_train_with_mag, X_rot], axis=0)
y_train_combined = np.concatenate([y_train, y_rot], axis=0)

In [53]:
# 6) compute mean/std on training combined and normalize train & test
mean = X_train_combined.mean(axis=(0,1))   # shape (8,)
std  = X_train_combined.std(axis=(0,1))    # shape (8,)

In [54]:
def normalize(X, mean, std):
    return (X - mean.reshape(1,1,-1)) / std.reshape(1,1,-1)

In [55]:
X_train_norm = normalize(X_train_combined, mean, std)
X_test_norm  = normalize(X_test_raw8, mean, std)  # use train stats for test normalization

# Now X_train_norm, y_train_combined are ready for training
# X_test_norm, y_test for evaluation
print("Shapes:", X_train_norm.shape, y_train_combined.shape, X_test_norm.shape, y_test.shape)

Shapes: (18078, 128, 8) (18078,) (2260, 128, 8) (2260,)


In [56]:
# 1) basic shapes
print("X_train:", X_train_norm.shape, " y_train:", y_train_combined.shape)
print("X_test: ", X_test_norm.shape,  " y_test: ", y_test.shape)

# 2) class distribution
import numpy as np
unique, counts = np.unique(y_train_combined, return_counts=True)
print("Train class counts:", dict(zip(unique, counts)))
unique, counts = np.unique(y_test, return_counts=True)
print("Test  class counts:", dict(zip(unique, counts)))

# 3) normalization correctness (means ~0, std ~1 on training)
print("Train mean per channel:", X_train_norm.mean(axis=(0,1)))
print("Train std  per channel:",  X_train_norm.std(axis=(0,1)))


X_train: (18078, 128, 8)  y_train: (18078,)
X_test:  (2260, 128, 8)  y_test:  (2260,)
Train class counts: {np.int64(0): np.int64(7286), np.int64(1): np.int64(2578), np.int64(2): np.int64(2948), np.int64(3): np.int64(2864), np.int64(4): np.int64(794), np.int64(5): np.int64(1608)}
Test  class counts: {np.int64(0): np.int64(910), np.int64(1): np.int64(366), np.int64(2): np.int64(313), np.int64(3): np.int64(390), np.int64(4): np.int64(85), np.int64(5): np.int64(196)}
Train mean per channel: [ 1.32592158e-14 -4.19013493e-16 -7.89756852e-16 -1.86143371e-16
 -3.66335349e-17 -5.43155703e-17  2.81294800e-14  8.11956018e-15]
Train std  per channel: [1. 1. 1. 1. 1. 1. 1. 1.]


In [57]:
model = Sequential([
    Conv1D(filters=8, kernel_size=5, activation='relu', padding='same',
           input_shape=(128, 8)),
    Conv1D(filters=16, kernel_size=5, activation='relu', padding='same'),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(6, activation='softmax')   # 6 classes
])

C:\Users\Fahad\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [58]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 128, 8)              │             328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 128, 16)             │             656 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 16)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 16)                  │             272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │             102 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,358 (5.30 KB)

 Trainable params: 1,358 (5.30 KB)

 Non-trainable params: 0 (0.00 B)

In [59]:
# 4) train
history = model.fit(
    X_train_norm, y_train_combined,
    validation_data=(X_test_norm, y_test),
    epochs=20,
    batch_size=32,
    verbose=1
)

Epoch 1/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.6642 - loss: 0.8154 - val_accuracy: 0.8000 - val_loss: 0.4769
Epoch 2/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7538 - loss: 0.6017 - val_accuracy: 0.8770 - val_loss: 0.4266
Epoch 3/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7765 - loss: 0.5662 - val_accuracy: 0.9058 - val_loss: 0.3500
Epoch 4/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7834 - loss: 0.5420 - val_accuracy: 0.8960 - val_loss: 0.3411
Epoch 5/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.7878 - loss: 0.5328 - val_accuracy: 0.9022 - val_loss: 0.3270
Epoch 6/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.7895 - loss: 0.5250 - val_accuracy: 0.9133 - val_loss: 0.3120
Epoch 7/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7889 - loss: 0.5221 - val_accuracy: 0.9035 - val_loss: 0.3196
Epoch 8/20
565/565 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.7917 - loss: 0.5157 - val_ac

In [61]:
val_loss, val_acc = model.evaluate(X_test_norm, y_test)
print("Validation Accuracy:", val_acc)

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9146 - loss: 0.2820
Validation Accuracy: 0.914601743221283


In [62]:
loss, acc = model.evaluate(X_test_norm, y_test)
print("Final test accuracy:", acc)

71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9146 - loss: 0.2820
Final test accuracy: 0.914601743221283


In [63]:
# Predict probabilities
y_pred_prob = model.predict(X_test_norm)

# Convert to class labels
y_pred = y_pred_prob.argmax(axis=1)


71/71 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [64]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1-Score :", f1)


Accuracy : 0.9146017699115044
Precision: 0.9066251444017406
Recall   : 0.9146017699115044
F1-Score : 0.9043582795727916
